# Numerai 13F-HR Data Updater (Robust XML Version)

このノートブックは、SEC EDGAR APIからNumerai GP LLC (CIK: 0001668527) の最新の13F-HR提出データを取得し、Googleドライブ上の `History_Numerai_13F-HR.xlsx` に追記します。

In [ ]:
# 必要なライブラリのインストール
!pip install requests pandas beautifulsoup4 openpyxl lxml -q

import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
from google.colab import drive
import re

# --- 設定 ---
CIK = "0001668527"
USER_AGENT = "Kei Sanada sdk7777@gmail.com"
SEC_HEADERS = {"User-Agent": USER_AGENT}

# Googleドライブをマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Excelファイルのパス（環境に合わせて書き換えてください）
EXCEL_PATH = "/content/drive/MyDrive/History_Numerai_13F-HR.xlsx"

### 1. 最新の13F-HR提出物を特定する

In [ ]:
def get_latest_13f_submission(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    response = requests.get(url, headers=SEC_HEADERS)
    response.raise_for_status()
    data = response.json()
    
    recent_filings = data['filings']['recent']
    for i, form in enumerate(recent_filings['form']):
        if form == '13F-HR':
            acc_num = recent_filings['accessionNumber'][i]
            report_date = recent_filings['reportDate'][i]
            return acc_num.replace('-', ''), report_date
    return None, None

acc_num_clean, latest_report_date = get_latest_13f_submission(CIK)
if acc_num_clean:
    print(f"最新の提出物を発見: Accession No: {acc_num_clean}, 報告基準日: {latest_report_date}")
else:
    print("13F-HR フォームが見つかりませんでした。")

### 2. XMLデータを取得してパースする

In [ ]:
def fetch_13f_data(cik, acc_num_clean, report_date):
    # 提出物ディレクトリのファイルリストを取得して XML ファイルを探す
    # アクセッション番号にはハイフンを入れた形式が必要な場合があるため調整
    # https://www.sec.gov/Archives/edgar/data/1668527/000121465924009228/index.json
    # (ハイフンなしディレクトリに index.json は存在する)
    
    base_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num_clean}"
    index_url = f"{base_url}/index.json"
    
    print(f"提出物インデックスを取得中: {index_url}")
    idx_resp = requests.get(index_url, headers=SEC_HEADERS)
    idx_resp.raise_for_status()
    
    # 提出物に含まれるファイルリストから "information table" を探す
    files = idx_resp.json()['directory']['item']
    xml_file = None
    
    # 優先順位: informationtable.xml -> *[0-9].xml (データテーブルは通常数字で終わる)
    for f in files:
        fname = f['name'].lower()
        if 'informationtable' in fname and fname.endswith('.xml'):
            xml_file = f['name']
            break
    
    if not xml_file:
        # 見つからない場合、数字を含むXML（データテーブルの可能性が高い）を探す
        for f in files:
            if f['name'].endswith('.xml') and not f['name'].startswith('primary_doc'):
                xml_file = f['name']
                break

    if not xml_file:
        raise FileNotFoundError("情報テーブルのXMLファイルが見つかりませんでした。")

    xml_url = f"{base_url}/{xml_file}"
    print(f"XMLを取得中: {xml_url}")
    response = requests.get(xml_url, headers=SEC_HEADERS)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'xml')
    info_table = []
    
    # 13F のXMLスキーマ（名前空間）に対応した抽出
    for info in soup.find_all(['infoTable', 'ns1:infoTable']):
        # 安全にテキストを取得するためのヘルパー
        def get_txt(tags):
            node = info.find(tags)
            return node.text if node else ''

        row = {
            'ISSUER NAME': get_txt(['nameOfIssuer', 'ns1:nameOfIssuer']),
            'TITLE OF CLASS': get_txt(['titleOfClass', 'ns1:titleOfClass']),
            'CUSIP': get_txt(['cusip', 'ns1:cusip']),
            'VALUE (x$1000)': int(get_txt(['value', 'ns1:value']) or 0),
            'SHRS OR PRN AMT': int(get_txt(['sshPrnamt', 'ns1:sshPrnamt']) or 0),
            'SH/ PRN': get_txt(['sshPrnamtType', 'ns1:sshPrnamtType']),
            'PUT/ CALL': get_txt(['putCall', 'ns1:putCall']),
            'INVESTMENT DISCRETION': get_txt(['investmentDiscretion', 'ns1:investmentDiscretion']),
            'OTHER MANAGER': get_txt(['otherManager', 'ns1:otherManager']),
            'SOLE': int(get_txt(['sole', 'ns1:sole']) or 0),
            'SHARED': int(get_txt(['shared', 'ns1:shared']) or 0),
            'NONE': int(get_txt(['none', 'ns1:none']) or 0),
            'Report Date': report_date
        }
        info_table.append(row)
    
    return pd.DataFrame(info_table)

if acc_num_clean:
    try:
        new_df = fetch_13f_data(CIK, acc_num_clean, latest_report_date)
        print(f"{len(new_df)} 件のデータを取得しました。")
    except Exception as e:
        print(f"エラーが発生しました: {e}")

### 3. GoogleドライブのExcelを更新する

In [ ]:
if 'new_df' in locals() and not new_df.empty:
    if os.path.exists(EXCEL_PATH):
        existing_df = pd.read_excel(EXCEL_PATH)
        if latest_report_date in existing_df['Report Date'].astype(str).unique():
            print(f"警告: 報告日 {latest_report_date} のデータは既にExcel内に存在します。")
        else:
            updated_df = pd.concat([existing_df, new_df], ignore_index=True)
            updated_df.to_excel(EXCEL_PATH, index=False)
            print(f"成功: {EXCEL_PATH} に {len(new_df)} 件のデータを追記しました。")
    else:
        new_data_dir = os.path.dirname(EXCEL_PATH)
        if not os.path.exists(new_data_dir):
            os.makedirs(new_data_dir)
        new_df.to_excel(EXCEL_PATH, index=False)
        print(f"成功: {EXCEL_PATH} を新規作成しました。")
else:
    print("エラー: 取得されたデータがないか、パースに失敗しました。")